In [15]:
import os
import requests
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

API_KEY = "YOUR_API_KEY"
IMAGE_SIZE = "224x224"
ZOOM = 18
MAP_TYPE = "satellite"
MAX_WORKERS = 10  

def fetch_image(lat, lon, save_path):
    url = (
        f"https://maps.googleapis.com/maps/api/staticmap?"
        f"center={lat},{lon}&zoom={ZOOM}&size={IMAGE_SIZE}"
        f"&maptype={MAP_TYPE}&key={API_KEY}"
    )
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            with open(save_path, "wb") as f:
                f.write(response.content)
    except Exception:
        pass  # silent fail

def fetch_dataset_images(csv_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)

    tasks = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for _, row in df.iterrows():
            img_path = os.path.join(output_dir, f"{row['id']}.png")
            if not os.path.exists(img_path):
                tasks.append(
                    executor.submit(
                        fetch_image,
                        row["lat"],
                        row["long"],
                        img_path
                    )
                )

        for _ in tqdm(as_completed(tasks), total=len(tasks)):
            pass

if __name__ == "__main__":
    fetch_dataset_images("test2(test(1)).csv", "data/images/train")
    fetch_dataset_images("train(1)(train(1)).csv", "data/images/test")


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16209/16209 [1:23:16<00:00,  3.24it/s]
